# 01 — Khám phá dataset

## Mục tiêu
Kiểm tra bộ câu hỏi suy luận tiếng Việt trước khi khóa pilot hoặc test. Kế hoạch dự kiến có 40 câu pilot và 300 câu test, với bốn nhóm chủ đề và ba mức độ khó. Chạy notebook riêng cho mỗi tập.

## Phương pháp và nguồn dữ liệu
Mở notebook trong bản sao dự án trên Colab hoặc Jupyter, đặt thư mục làm việc ở gốc dự án hoặc trong `notebooks/`, rồi cài các gói trong `requirements.txt`. Đặt `DATASET_PATH` thành đường dẫn CSV hoặc JSONL thật và cập nhật thông tin nguồn, phiên bản và tập dữ liệu.

**Chế độ mặc định dùng dữ liệu GIẢ chỉ để kiểm thử pipeline.** Fixture chứa placeholder và lỗi cố ý. Không đưa các số liệu từ fixture vào báo cáo thực nghiệm.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from IPython.display import display

import runpy

cwd = Path.cwd()
helper = next((base / 'src/r1_project_root.py' for base in (cwd, *cwd.parents)
               if (base / 'src/r1_project_root.py').is_file()), None)
if helper is None:
    matches = list(cwd.glob('*/src/r1_project_root.py'))
    if len(matches) != 1:
        raise FileNotFoundError('Không xác định được repository. Hãy clone dự án hoặc chuyển cwd vào bản clone.')
    helper = matches[0]
project_root = runpy.run_path(str(helper))['find_project_root']()
sys.path.insert(0, str(project_root))
from src.dataset import dataset_distribution, load_dataset, validate_dataset

DATASET_PATH = None  # Ví dụ khi có dữ liệu thật: 'data/test.jsonl'
DATASET_SOURCE = None
DATASET_VERSION = None
DATASET_SPLIT = None  # 'pilot' hoặc 'test'
TOPICS = ('toán có lời văn', 'logic và ràng buộc', 'suy luận từ đoạn văn',
          'suy luận định lượng thực tế')

fixture_dir = (project_root / 'data/fixtures').resolve()
path = (project_root / DATASET_PATH).resolve() if DATASET_PATH else fixture_dir / 'SYNTHETIC_pipeline_check.csv'
is_fixture = path.is_relative_to(fixture_dir)
if is_fixture:
    print('DỮ LIỆU GIẢ — chỉ dùng kiểm thử pipeline')
else:
    print(f'Nguồn: {DATASET_SOURCE}; phiên bản: {DATASET_VERSION}; tập: {DATASET_SPLIT}')
dataset = load_dataset(path)
print(f'Tệp: {path.name} | Số bản ghi: {len(dataset)}')

## Kiểm tra chất lượng dữ liệu
Kiểm tra trường bắt buộc, giá trị thiếu, kiểu dữ liệu, nhãn và bản ghi trùng. Số thứ tự bản ghi bắt đầu từ 1 và bỏ qua dòng trống. Một bản ghi có thể có nhiều lỗi. Phát hiện đề trùng sau khi chuẩn hóa Unicode NFC, chữ hoa/thường và khoảng trắng. Cần rà soát thủ công các biến thể đổi số liệu hoặc cách diễn đạt.

In [ ]:
issues = validate_dataset(dataset, allowed_topics=('fixture',) if is_fixture else TOPICS)
print(f'Số phát hiện: {len(issues)} | Số bản ghi có lỗi: {issues["row"].nunique()}')
if issues.empty:
    print('Không phát hiện lỗi theo các quy tắc đã cấu hình.')
else:
    display(issues.groupby(['code', 'field'], dropna=False).size().rename('count').reset_index())
    display(issues)

## Phân phối chủ đề và độ khó
Các bảng và biểu đồ đếm mọi bản ghi, kể cả những bản ghi cần rà soát. Nhãn thiếu hoặc sai kiểu xuất hiện trong nhóm thiếu. Nhãn văn bản lạ được hiển thị để phát hiện các nhóm ngoài danh mục.

In [ ]:
for field, title in [('topic', 'Chủ đề'), ('difficulty', 'Độ khó')]:
    table = dataset_distribution(dataset, field)
    display(table)
    if table.empty:
        continue
    labels = table[field].fillna('(thiếu hoặc sai kiểu)')
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.barh(range(len(table)), table['count'])
    ax.set_yticks(range(len(table)), labels=labels)
    prefix = 'DỮ LIỆU GIẢ — ' if is_fixture else ''
    ax.set(title=prefix + title, xlabel='Số bản ghi')
    fig.tight_layout()
    plt.show()

display(dataset_distribution(dataset, 'topic', 'difficulty'))

## Kết quả và nhận xét
- Fixture được thiết kế để làm lộ một đề trùng, một đáp án thiếu và một nhãn độ khó ngoài danh mục. Đây là kỳ vọng kiểm thử, không phải phát hiện về dataset nghiên cứu.
- Khi dùng dữ liệu thật, ghi cỡ mẫu, số phát hiện theo loại và những nhóm lệch so với kế hoạch. Đề trùng có thể làm một kiểu bài được tính nhiều lần. Nhãn thiếu hoặc không nhất quán có thể làm sai phân tích theo nhóm.
- Bảng tần suất chỉ cho thấy độ phủ của các nhãn đã gán. Hãy kiểm tra chất lượng đáp án bằng cách giải lại đề và kiểm tra trùng họ câu hỏi giữa pilot và test trước khi khóa dữ liệu.
- Chưa có kết quả dataset thật để kết luận về mức độ cân bằng hoặc chất lượng.